<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/KU_Parking_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q git+https://github.com/KarAnalytics/llm_cascade.git


  Preparing metadata (setup.py) ... done


# KU Parking Assistant: A Simple Tool-Using Agent

This notebook builds a **small agent** that helps you find parking near any building at the University of Kansas.

**What the agent does:**
- You ask: *"Where can I park near the business school?"*
- The agent uses **tools** (Python functions) to look up the building's coordinates, find nearby parking lots, calculate distances, and return a ranked list
- For each parking lot, it tells you the **color** (Blue / Yellow / Gold / Red / Visitor) and gives you a **Google Maps pin** link you can click to navigate

**What's new vs. the other agent notebooks?**
- We use **real deterministic tools** (distance math, data lookups) instead of a pure text-in/text-out workflow
- The LLM decides **when** to call which tool, but it doesn't do the math itself — it delegates to Python
- This is a **pure workflow**, not a chatbot: one input → agent reasoning → one output. Much simpler than the MultiAgent_DB chatflow.

**Teaching goal:** Show the difference between:
- **Fuzzy LLM reasoning** (good for language understanding: "business school" → "Capitol Federal Hall")
- **Precise tool execution** (good for deterministic tasks: distance math, sorting, data lookup)

A well-designed agent lets each side do what it's best at.

**Note on data:** The real KU parking map is at https://parking.ku.edu/sites/parking/files/documents/parkingmap.pdf. For this demo we use a **small illustrative dataset** with approximate coordinates so the notebook runs standalone. Students can later replace our dataset with real data parsed from the PDF.


## Imports and LLM Setup

We use `llm_cascade` for automatic LLM provider fallback, and Python's built-in `math` module for the distance calculations. No external APIs needed.


In [2]:
import math
import json as _json
import re as _re

from llm_cascade import get_cascade

llm = get_cascade()


LLM Cascade - available providers:
  + Gemini           model=gemini-2.5-flash
  + Ollama           model=kimi-k2.5:cloud
  + Groq             model=llama-3.3-70b-versatile
  + HuggingFace      model=meta-llama/Llama-3.3-70B-Instruct
  + Cohere           model=command-a-03-2025
  + OpenRouter       model=meta-llama/llama-3.3-70b-instruct:free
  + OpenAI           model=gpt-4o-mini
Not configured (skipped):
  - Grok (xAI)       (set XAI_API_KEY)


## Data: KU Buildings and Parking Lots

We hardcode two small datasets. In a real deployment you would read these from a database or parse them from the PDF.

**KU Buildings** — the destinations students might ask about. Coordinates are approximations of real locations.

**KU Parking Lots** — each has a name, coordinates, and a *color* indicating who can park there:
- **Blue** — student commuter permit
- **Yellow** — faculty / staff permit
- **Gold** — premium permit (close to central campus)
- **Red** — restricted / reserved
- **Visitor** — open to visitors, usually metered or short-term
- **Park & Ride** — free remote lot with shuttle service


In [3]:
# --- KU Buildings (approximate coordinates for illustration) ---
KU_BUILDINGS = {
    'Capitol Federal Hall (Business School)':        (38.953505, -95.249740),
    'Kansas Union':                                  (38.959200, -95.243500),
    'Allen Fieldhouse':                              (38.954306, -95.252394),
    'Strong Hall':                                   (38.958542, -95.247614),
    'Watson Library':                                (38.956621, -95.244787),
    'Anschutz Library':                              (38.957323, -95.249742),
    'Dyche Hall':                                    (38.958610, -95.243890),
    'Fraser Hall':                                   (38.957160, -95.243590),
    'Lied Center':                                   (38.954940, -95.262890),
    'Memorial Stadium':                              (38.963330, -95.246390),
    'Wescoe Hall':                                   (38.957430, -95.247830),
    'Spencer Museum of Art':                         (38.959634, -95.244569),
    'Ambler Student Recreation Fitness Center':      (38.952512, -95.247929),
}

# --- KU Parking Lots (approximate coordinates for illustration) ---
KU_PARKING_LOTS = [
    {'lot': 'Lot 3'                  , 'color': 'Blue'    , 'lat': 38.958942, 'lng': -95.247614},
    {'lot': 'Lot 5'                  , 'color': 'Yellow'  , 'lat': 38.95861, 'lng': -95.24441},
    {'lot': 'Lot 10'                 , 'color': 'Blue'    , 'lat': 38.956221, 'lng': -95.244787},
    {'lot': 'Lot 14'                 , 'color': 'Yellow'  , 'lat': 38.95716, 'lng': -95.24307},
    {'lot': 'Lot 16'                 , 'color': 'Visitor' , 'lat': 38.9592, 'lng': -95.24298},
    {'lot': 'Lot 18'                 , 'color': 'Yellow'  , 'lat': 38.95703, 'lng': -95.24783},
    {'lot': 'Lot 60'                 , 'color': 'Blue'    , 'lat': 38.96333, 'lng': -95.24691},
    {'lot': 'Lot 70'                 , 'color': 'Blue'    , 'lat': 38.953906, 'lng': -95.252394},
    {'lot': 'Lot 71'                 , 'color': 'Yellow'  , 'lat': 38.953666, 'lng': -95.252394},
    {'lot': 'Lot 90'                 , 'color': 'Gold'    , 'lat': 38.952512, 'lng': -95.247929},
    {'lot': 'Lot 91'                 , 'color': 'Gold'    , 'lat': 38.959634, 'lng': -95.244569},
    {'lot': 'Lot 118'                , 'color': 'Gold'    , 'lat': 38.953505, 'lng': -95.24974},
    {'lot': 'Lot 300A-D'             , 'color': 'Gold'    , 'lat': 38.95494, 'lng': -95.26237},
    {'lot': 'Lot 300E-G'             , 'color': 'Gold'    , 'lat': 38.95494, 'lng': -95.26289},
    {'lot': 'Allen Fieldhouse Garage', 'color': 'Gold'    , 'lat': 38.954306, 'lng': -95.252394},
]

print(f'Loaded {len(KU_BUILDINGS)} buildings and {len(KU_PARKING_LOTS)} parking lots.')


Loaded 13 buildings and 15 parking lots.


## The Tools (plain Python functions)

Three simple tools the agent can call. Notice there is **no LLM inside these** — they're pure deterministic Python. The LLM will decide *when* to call them based on the user's question.

| Tool | What it does |
|---|---|
| `list_ku_buildings()` | Returns the list of available buildings the user can ask about |
| `find_parking_near_building(name, top_k)` | Finds the top-K nearest parking lots to a building, with distances and Google Maps URLs |
| `get_parking_colors_legend()` | Explains what each parking color means |

The distance calculation uses the **Haversine formula** — accurate great-circle distance between two lat/lng points. We let Python do the math; the LLM just decides when to call it.


In [4]:
def _haversine_miles(lat1, lng1, lat2, lng2):
    '''Great-circle distance between two lat/lng points, in miles.'''
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lng2 - lng1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def _google_maps_pin(lat, lng):
    '''Return a clickable Google Maps pin URL for a coordinate.'''
    return f'https://www.google.com/maps?q={lat},{lng}'


def _match_building(query):
    '''Fuzzy-match a building name. Accepts partial/lowercase queries like "business school".'''
    q = query.lower().strip()
    # Exact match first
    for name in KU_BUILDINGS:
        if name.lower() == q:
            return name
    # Substring match
    for name in KU_BUILDINGS:
        if q in name.lower():
            return name
    # Reverse substring (user typed a longer phrase that contains the building name)
    for name in KU_BUILDINGS:
        short_key = name.split('(')[0].strip().lower()
        if short_key in q:
            return name
    return None


# ---- Tools the agent can call ----

def list_ku_buildings(_=''):
    '''Return the list of KU buildings available for parking lookup.'''
    return chr(10).join(f'- {name}' for name in KU_BUILDINGS.keys())


def find_parking_near_building(building_name, top_k=5, max_distance_miles=2.0):
    '''Find nearest parking lots within max_distance_miles of the building, top-K results.'''
    matched = _match_building(building_name)
    if matched is None:
        return (
            f'No KU building matches "{building_name}". '
            f'Use list_ku_buildings to see available options.'
        )

    target_lat, target_lng = KU_BUILDINGS[matched]

    # Compute distance to every lot and sort
    ranked = []
    for lot in KU_PARKING_LOTS:
        dist = _haversine_miles(target_lat, target_lng, lot['lat'], lot['lng'])
        ranked.append({
            'lot': lot['lot'],
            'color': lot['color'],
            'distance_miles': round(dist, 3),
            'walk_minutes': round(dist * 20),  # ~3 mph walking speed
            'google_maps': _google_maps_pin(lot['lat'], lot['lng']),
        })

    ranked.sort(key=lambda r: r['distance_miles'])

    # Filter to lots within max_distance_miles of the destination
    within_range = [r for r in ranked if r['distance_miles'] <= max_distance_miles]

    if not within_range:
        return (
            f'No parking lots within {max_distance_miles} miles of {matched}. '
            f'Try a different building or increase the radius.'
        )

    header = (
        f'Parking lots within {max_distance_miles} miles of {matched} '
        f'(top {top_k}, ranked by distance):' + chr(10)
    )
    lines = [header]
    for i, r in enumerate(within_range[:top_k], start=1):
        lines.append(
            f"  {i}. {r['lot']} ({r['color']}) - "
            f"{r['distance_miles']} mi ({r['walk_minutes']} min walk) "
            f"-> {r['google_maps']}"
        )
    return chr(10).join(lines)


def get_parking_colors_legend(_=''):
    '''Explain what each parking color means at KU.'''
    return (
        'KU parking color legend:' + chr(10)
        + '- Blue: student commuter permit' + chr(10)
        + '- Yellow: faculty/staff permit' + chr(10)
        + '- Gold: premium permit (close to central campus)' + chr(10)
        + '- Red: restricted/reserved' + chr(10)
        + '- Visitor: open to visitors (often metered or short-term)' + chr(10)
        + '- Park & Ride: free remote lot with shuttle service'
    )


# Register tools for the agent
TOOLS = {
    'list_ku_buildings':            list_ku_buildings,
    'find_parking_near_building':   find_parking_near_building,
    'get_parking_colors_legend':    get_parking_colors_legend,
}

print('Tools registered:', list(TOOLS.keys()))


Tools registered: ['list_ku_buildings', 'find_parking_near_building', 'get_parking_colors_legend']


## Test the Tools (no LLM yet)

Before wiring up the agent, let's call the tools directly to make sure they work. This is also how you debug tool logic without burning LLM calls.


In [5]:
# Call one tool directly
print(find_parking_near_building('business school', top_k=5))
print()
print(find_parking_near_building('Kansas Union', top_k=3))


Parking lots within 2.0 miles of Capitol Federal Hall (Business School) (top 5, ranked by distance):

  1. Lot 118 (Gold) - 0.0 mi (0 min walk) -> https://www.google.com/maps?q=38.953505,-95.24974
  2. Lot 90 (Gold) - 0.119 mi (2 min walk) -> https://www.google.com/maps?q=38.952512,-95.247929
  3. Lot 71 (Yellow) - 0.143 mi (3 min walk) -> https://www.google.com/maps?q=38.953666,-95.252394
  4. Lot 70 (Blue) - 0.145 mi (3 min walk) -> https://www.google.com/maps?q=38.953906,-95.252394
  5. Allen Fieldhouse Garage (Gold) - 0.153 mi (3 min walk) -> https://www.google.com/maps?q=38.954306,-95.252394

Parking lots within 2.0 miles of Kansas Union (top 3, ranked by distance):

  1. Lot 16 (Visitor) - 0.028 mi (1 min walk) -> https://www.google.com/maps?q=38.9592,-95.24298
  2. Lot 5 (Yellow) - 0.064 mi (1 min walk) -> https://www.google.com/maps?q=38.95861,-95.24441
  3. Lot 91 (Gold) - 0.065 mi (1 min walk) -> https://www.google.com/maps?q=38.959634,-95.244569


## The Agent (LLM that calls the tools)

We use the same **prompt-based ReAct loop** as the LangChain agent notebook — no `bind_tools`, no special API features, works with any LLM. The LLM outputs a JSON action like `{"tool": "find_parking_near_building", "input": "business school"}` and we parse and execute it.

The system prompt describes the available tools and tells the LLM how to use them. On each turn, the agent either calls a tool or returns a final answer.


In [6]:
AGENT_SYSTEM = (
    'You are a helpful KU parking assistant. You have these tools:' + chr(10)
    + '- list_ku_buildings(): returns the list of KU buildings available.' + chr(10)
    + '- find_parking_near_building(name): returns the top 5 nearest parking lots to a building, '
    + 'with color, distance, walk time, and Google Maps pin.' + chr(10)
    + '- get_parking_colors_legend(): explains the KU parking color codes.' + chr(10) + chr(10)
    + 'To use a tool, reply with ONLY this JSON (nothing else):' + chr(10)
    + '{"tool": "tool_name", "input": "the input string"}' + chr(10) + chr(10)
    + 'After receiving a tool result, write a helpful final answer for the user that includes '
    + 'the ranked parking list, color meanings, and clickable Google Maps links. '
    + 'If the user mentions a building informally (e.g., "business school"), pass that phrase '
    + 'directly to find_parking_near_building — its fuzzy matcher will handle it.'
)


def run_parking_agent(question, verbose=True):
    '''Simple ReAct-style loop: LLM decides which tool to call, we execute it, feed result back.'''
    messages = [
        {'role': 'system', 'content': AGENT_SYSTEM},
        {'role': 'user', 'content': question},
    ]

    # Flatten into a single prompt with conversation history
    def flatten(msgs):
        parts = []
        for m in msgs:
            parts.append(f"{m['role'].upper()}: {m['content']}")
        return chr(10).join(parts)

    for step in range(5):  # max 5 tool rounds
        prompt = flatten(messages[1:])  # send everything except system (passed separately)
        response = llm.generate(prompt, system_prompt=AGENT_SYSTEM)
        reply = response.text.strip()

        # Try to parse a tool call
        tool_call = None
        m = _re.search(r'\{[^{}]+\}', reply)
        if m:
            try:
                parsed = _json.loads(m.group())
                if 'tool' in parsed and 'input' in parsed:
                    tool_call = parsed
            except _json.JSONDecodeError:
                pass

        if tool_call is None:
            # No tool call -- this is the final answer
            if verbose:
                print(f'  [Step {step+1}] Final answer' + chr(10))
            return reply

        # Execute the requested tool
        tool_name = tool_call['tool']
        tool_input = tool_call['input']
        tool_fn = TOOLS.get(tool_name)
        if tool_fn:
            if verbose:
                print(f'  [Step {step+1}] Calling: {tool_name}({tool_input})')
            result = tool_fn(tool_input)
            if verbose:
                preview = str(result).replace(chr(10), ' | ')[:150]
                print(f'              Result: {preview}...')
        else:
            result = f'Unknown tool: {tool_name}'

        messages.append({'role': 'assistant', 'content': reply})
        messages.append({'role': 'user', 'content': f'Tool result: {result}'})

    return 'Agent exceeded maximum steps.'


print('Parking agent ready.')


Parking agent ready.


## Ask the Agent Some Questions

Try different destinations. The agent should:
1. Figure out which tool to call
2. Call `find_parking_near_building` with the user's phrase
3. Format a helpful response with the ranked list and Google Maps links


In [7]:
from IPython.display import display, Markdown

questions = [
    'Where can I park near the business school?',
    # 'I need parking near the student union. What are my options?',
    # 'What colors of parking are available at KU and what do they mean?',
    # 'I am visiting KU and want to park near Allen Fieldhouse. Where should I go?',
]

for q in questions:
    display(Markdown('---'))
    display(Markdown(f'**Q:** {q}'))
    answer = run_parking_agent(q)
    # Render as Markdown so Google Maps URLs become clickable links.
    # Colab auto-linkifies any URL inside a Markdown-rendered cell.
    display(Markdown(answer))


---

**Q:** Where can I park near the business school?

  [Response from Gemini / gemini-2.5-flash]
  [Step 1] Calling: find_parking_near_building(business school)
              Result: Parking lots within 2.0 miles of Capitol Federal Hall (Business School) (top 5, ranked by distance): |  |   1. Lot 118 (Gold) - 0.0 mi (0 min walk) ->...
  [Response from Gemini / gemini-2.5-flash]
  [Step 2] Calling: get_parking_colors_legend()
              Result: KU parking color legend: | - Blue: student commuter permit | - Yellow: faculty/staff permit | - Gold: premium permit (close to central campus) | - Red...
  [Response from Gemini / gemini-2.5-flash]
  [Step 3] Final answer



Here are the top 5 nearest parking lots to Capitol Federal Hall (Business School), along with their color codes, distances, and estimated walk times:

1.  **Lot 118** (Gold) - 0.0 mi (0 min walk) - [Google Maps Pin](https://www.google.com/maps?q=38.953505,-95.24974)
2.  **Lot 90** (Gold) - 0.119 mi (2 min walk) - [Google Maps Pin](https://www.google.com/maps?q=38.952512,-95.247929)
3.  **Lot 71** (Yellow) - 0.143 mi (3 min walk) - [Google Maps Pin](https://www.google.com/maps?q=38.953666,-95.252394)
4.  **Lot 70** (Blue) - 0.145 mi (3 min walk) - [Google Maps Pin](https://www.google.com/maps?q=38.953906,-95.252394)
5.  **Allen Fieldhouse Garage** (Gold) - 0.153 mi (3 min walk) - [Google Maps Pin](https://www.google.com/maps?q=38.954306,-95.252394)

**KU Parking Color Legend:**

*   **Blue:** student commuter permit
*   **Yellow:** faculty/staff permit
*   **Gold:** premium permit (close to central campus)
*   **Red:** restricted/reserved
*   **Visitor:** open to visitors (often metered or short-term)
*   **Park & Ride:** free remote lot with shuttle service

Please make sure you have the appropriate permit for the lot you choose!

---

## How to Build the Same Thing in Dify (multi-node workflow with real tools)

A proper Dify workflow uses **alternating tool and LLM nodes**, matching the architecture you see in typical agentic workflow diagrams. Each tool node fetches data, each LLM node reasons about what to do next with that data.

### The workflow architecture

```
Start (user types a KU building name)
  |
  v
[TOOL #1] Knowledge Retrieval - KU_Buildings KB
  | (acts as the "access Google Maps" step: returns the building's lat/lng)
  v
[LLM #1] Parse the building coordinates
  | (extracts a clean {lat, lng} from the retrieved text)
  v
[TOOL #2] Knowledge Retrieval - KU_Parking_Lots KB
  | (acts as the "access KU map" step: returns all parking lot data)
  v
[LLM #2] Compute distances, filter to 2 miles, rank top 5, format output
  | (does the math approximately using the coordinates)
  v
End (returns formatted parking list with Google Maps pin URLs)
```

Five processing nodes: 2 Knowledge Retrieval (tool) nodes + 2 LLM nodes + Start/End. No Code nodes, no custom APIs, no Python.

### Why Knowledge Retrieval instead of HTTP Request?

In Dify's workflow builder, a "tool" is any node that fetches external data. Knowledge Retrieval nodes are the easiest kind of tool because:
- You upload the data once (buildings + parking lots as markdown files)
- Dify chunks, embeds, and indexes it automatically
- Retrieval is just a vector search — fast and free
- No API keys, no rate limits, no billing

An alternative would be an **HTTP Request** node calling Nominatim (OpenStreetMap's free geocoding API), but Nominatim is rate-limited and doesn't know anything about specific KU building nicknames like "business school". Our pre-uploaded Knowledge Base is more reliable for a classroom demo.

### Step-by-step setup

#### Part A: upload the two data files as Knowledge Bases

The notebook directory includes two markdown files at `Agentic/dify_data/`:
- `ku_buildings.md` - one section per building with coordinates
- `ku_parking_lots.md` - one section per parking lot with color and coordinates

In Dify:

1. Go to [cloud.dify.ai](https://cloud.dify.ai) -> **Knowledge** -> **Create Knowledge**
2. Upload `ku_buildings.md`. Name the KB `KU_Buildings`.
3. On the chunking screen, pick **Custom** chunking:
   - Chunk delimiter: `\n# ` (hash + space — this splits on every heading)
   - Max chunk length: 500 tokens
   - This ensures each building becomes its own retrievable chunk.
4. Indexing: **High Quality** (vector embeddings).
5. Click **Save & Process**. Wait ~30 seconds.
6. Repeat: create a second Knowledge Base called `KU_Parking_Lots` and upload `ku_parking_lots.md` with the same chunking settings.

#### Part B: build the workflow

1. **Studio** -> **Create App** -> **Workflow** (not Chatflow).
2. Name it `KU Parking Assistant`.
3. On the canvas, click the **Start** node:
   - Add input variable: name = `destination`, type = Text, label = "Which KU building are you visiting?"

4. **Add TOOL #1: Knowledge Retrieval (Buildings)**
   - Click **+** after Start -> **Knowledge Retrieval**
   - Query variable: `Start.destination`
   - Knowledge: select `KU_Buildings`
   - Rename this node to `Retrieve Building`

5. **Add LLM #1: Parse Building Coordinates**
   - Click **+** after Retrieve Building -> **LLM**
   - Model: your preferred (gpt-4o-mini, gemini-2.5-flash, etc.)
   - **System prompt**:
     ```
     You are a data extractor. The user input contains text describing a KU building.
     Extract the building name and its latitude/longitude coordinates.
     Reply with ONLY this JSON format (nothing else):
     {"building": "full name", "lat": 38.9573, "lng": -95.2519}
     ```
   - **User prompt**:
     ```
     Retrieved building information:
     {{#Retrieve_Building.result#}}
     
     User query: {{#Start.destination#}}
     ```
   - Rename this node to `Parse Coords`

6. **Add TOOL #2: Knowledge Retrieval (Parking Lots)**
   - Click **+** after Parse Coords -> **Knowledge Retrieval**
   - Query variable: `Start.destination` (or a fixed query like "parking lots")
   - Knowledge: select `KU_Parking_Lots`
   - In the node settings, set **Top K** to 15 (to retrieve all lots, since our KB is small)
   - Rename this node to `Retrieve Parking`

7. **Add LLM #2: Distance + Filter + Format**
   - Click **+** after Retrieve Parking -> **LLM**
   - **System prompt**:
     ```
     You are a parking assistant. You will receive:
     1. A destination building with its latitude and longitude (as JSON).
     2. A list of KU parking lots with their coordinates and colors.
     
     Your job:
     - Compute the approximate great-circle distance in miles from the destination
       to each parking lot. For Lawrence KS, 0.001 degrees of latitude is about
       0.069 miles and 0.001 of longitude is about 0.054 miles.
     - Filter to lots within 2 miles of the destination.
     - Rank the remaining lots from nearest to farthest.
     - Return the top 5 as a numbered list with these fields for each:
       * Rank, lot name, permit color
       * Distance in miles (2 decimals) and approximate walk time (3 mph walking speed)
       * Google Maps pin URL in this exact format: https://www.google.com/maps?q=LAT,LNG
     - End with a short note explaining what the permit color of the closest lot means.
     - If no lots are within 2 miles, say so and suggest the Park & Ride shuttle.
     ```
   - **User prompt**:
     ```
     Destination: {{#Parse_Coords.text#}}
     
     Available parking lots:
     {{#Retrieve_Parking.result#}}
     
     Give me the top 5 parking lots within 2 miles.
     ```
   - Rename this node to `Rank and Format`

8. **Configure the End node**
   - Output variable: `Rank_and_Format.text`

9. **Publish** and click **Run** to test. Try: "business school", "Kansas Union", "Allen Fieldhouse".

### What each node does and why

| Node | Type | Job | Deterministic or LLM? |
|---|---|---|---|
| Start | Input | Collect user destination | - |
| Retrieve Building | Tool (Knowledge Retrieval) | Look up the building in the Buildings KB | Deterministic (vector search) |
| Parse Coords | LLM | Extract clean `{lat, lng}` from the retrieved text | LLM (structured extraction) |
| Retrieve Parking | Tool (Knowledge Retrieval) | Look up parking lots from the Parking KB | Deterministic (vector search) |
| Rank and Format | LLM | Compute distances, filter 2 miles, rank, format | LLM (approximate math + formatting) |
| End | Output | Return the formatted result | - |

Notice the **alternation**: Tool -> LLM -> Tool -> LLM. Each tool fetches fresh data, each LLM reasons about what to do with it. This is the standard pattern in Dify workflows and maps directly to the architecture diagram you described.

### Colab (real tools) vs. Dify (workflow tools) - the teaching contrast

| | Colab notebook | Dify workflow |
|---|---|---|
| How are tools defined? | Python functions | Dify nodes (Knowledge Retrieval, LLM, HTTP Request) |
| Who decides which tool to call? | The LLM, at each step of the agent loop | Fixed by the workflow diagram (same tools in the same order) |
| Distance math | Exact (Haversine formula) | Approximate (LLM reasoning over coordinates) |
| Data source | Python dicts | Dify Knowledge Bases with vector retrieval |
| Flexibility | High — agent can call tools in any order | Low — flow is fixed, but predictable |
| Best for | Complex questions with branching reasoning | Simple, predictable pipelines with the same steps every time |

### When to pick each approach

- **Colab agent (this notebook):** when the agent needs to decide dynamically whether to call `list_buildings`, `find_parking_near_building`, or `get_parking_colors_legend` based on the question. Good for conversational or exploratory use cases.
- **Dify workflow (above):** when every query follows the same steps (lookup destination -> lookup parking -> rank). Good for a production app where you want consistent, debuggable behavior without per-query reasoning overhead.

### Limitations of the Dify version

- The LLM's distance math is **approximate**. For 15 lots in a small area it usually gets the top 3 right, but lots at very similar distances may be mis-ranked by 1-2 positions.
- If your Knowledge Base grows beyond a few hundred entries, the vector retrieval step may miss relevant items. At that scale you'd switch to HTTP Request calls to a real database.
- The 2-mile filter is enforced in the LLM prompt, not in code. A rogue LLM response could include a lot outside 2 miles. In production you'd add a **Template Transform** or **Parameter Extractor** node to enforce the filter programmatically.


## Exercises

- Add a new building (e.g., `'Budig Hall': (lat, lng)`) and a new parking lot, then rerun — notice you don't need to change any tool code.
- Add a fourth tool: `find_parking_by_color(color, near_building)` that filters to only lots of a specific permit color.
- Replace the hardcoded data with a real parse of the [KU parking map PDF](https://parking.ku.edu/sites/parking/files/documents/parkingmap.pdf) — this is where real data engineering begins.
- In Dify, try removing the coordinates from the system prompt and just listing lot names. How does answer quality change? This shows why giving the LLM structured numeric data matters.
- Compare the Colab and Dify outputs for the same query. How often do they agree on the top-3 ranking?
